In [103]:
import gc
import sys

# Disable automatic GC for this demonstration
# gc.disable()

gc.set_debug(gc.DEBUG_UNCOLLECTABLE)

# Create one list object.
numbers = [10, 20, 30]

# 'another' points to the same list.
another = numbers

print(numbers)
print(another)

print(f"Reference count of {another} is: {sys.getrefcount(another) - 1}")

[10, 20, 30]
[10, 20, 30]
Reference count of [10, 20, 30] is: 2


In [104]:
del numbers

print(f"Reference count of {another} after del is: {sys.getrefcount(another) - 1}")

Reference count of [10, 20, 30] after del is: 1


In [105]:
users = ["Alice", "Bob"]

print("Reference Count:", sys.getrefcount(users))

another = users

print("Reference Count:", sys.getrefcount(users))

del another

print("Reference Count:", sys.getrefcount(users))

Reference Count: 2
Reference Count: 3
Reference Count: 2


In [106]:
x = 10
print(id(x))

x = x + 1
print(id(x))

4358080968
4358081000


In [107]:
a = [1, 3, 6]
b = [1, 3, 6]

print(f"a == b -> {a == b}")
print(f"a is b -> {a is b}")

a == b -> True
a is b -> False


In [108]:
a = [[1, 2], [3, 4], [[1, [8]]]]
b = [[1, 2], [3, 4], [[1, [8]]]]
print(f"a == b -> {a == b}")
print(f"a is b -> {a is b}")

a == b -> True
a is b -> False


In [109]:
a = ([1, 2], (3, 4), [(1, [8])])
b = ([1, 2], (3, 4), [(1, [8])])
print(f"a == b -> {a == b}")
print(f"a is b -> {a is b}")

a == b -> True
a is b -> False


In [110]:
print(f"ID of None is: {id(None)}")
none_val = None

print(f"ID of {none_val} is: {id(none_val)}")

ID of None is: 4357794672
ID of None is: 4357794672


In [111]:
print(f"ID of 1 is: {id(1)}")
num = 1

print(f"ID of num is: {id(num)}")

ID of 1 is: 4358080680
ID of num is: 4358080680


In [136]:
print(f"ID of 'Hello' is: {id('Hello')}")
hello = "Hello"

print(f"ID of hello is: {id(hello)}")

li = ["Hello"]
print(f"ID of hello in list is: {id(li[0])}")

ID of 'Hello' is: 4565475344
ID of hello is: 4565475344
ID of hello in list is: 4565475344


In [ ]:
class Node:
    def __init__(self, name):
        self.name = name
        self.ref = None


node1 = Node("A")
node2 = Node("B")

print(f"node1 is node1 -> {node1 is node1}")
print(f"node1 is node1 -> {node1 == node1}")

# Create the cycle
node1.ref = node2
node2.ref = node1

print(f"Refernce count for Node1: {sys.getrefcount(node1) - 1}")
print(f"Refernce count for Node2: {sys.getrefcount(node2) - 1}")
# Remove external references
del node1
del node2


node1 is node1 -> True
node1 is node1 -> True
Refernce count for Node1: 2
Refernce count for Node2: 2


In [114]:
# Force a collection
unreachable_count = gc.collect()
print(f"Unreachable objects collected: {unreachable_count}")
# Output will typically be > 0, indicating the cycle was broken

# Re-enable GC
# gc.enable()

Unreachable objects collected: 3517


In [115]:
print(gc.get_threshold())

(2000, 10, 10)


In [116]:
# (gen0, gen1, gen2)
gc.set_threshold(10000, 5, 10)
print(gc.get_threshold())

(10000, 5, 10)


In [117]:
import gc

import objgraph


class WebRequest:
    def __init__(self, request_id):
        self.request_id = request_id
        # Simulating a large payload
        self.payload = [x for x in range(10000)]


# A global dictionary holding references
active_requests = {}


def handle_request(req_id):
    req = WebRequest(req_id)
    # MISTAKE: We store the request but forget to remove it later
    active_requests[req_id] = req


# Simulate requests
handle_request("req-001")
handle_request("req-002")

# 1. Show the most common types of objects currently in memory.
# This helps identify if a specific class is leaking.
print("--- Top Object Growth ---")
objgraph.show_growth(limit=3)

# 2. Grab a random instance of our WebRequest object from memory
requests_in_memory = objgraph.by_type("WebRequest")
if requests_in_memory:
    leaking_obj = requests_in_memory[0]

    # 3. Generate a visual graph of what is pointing to this object.
    # This generates a PNG file (requires Graphviz installed on the system).
    # In production, look at this graph to see that 'active_requests' is the culprit.
    print(f"\nGenerating backreference graph for {leaking_obj}...")
    objgraph.show_backrefs([leaking_obj], max_depth=3, filename="leak_graph.png")

--- Top Object Growth ---
function    23312    +23312
dict        17550    +17550
tuple       12593    +12593

Generating backreference graph for <__main__.WebRequest object at 0x10af9e7b0>...
Graph written to /var/folders/cb/by3krgzd6rzgcyscfhww7j180000gn/T/objgraph-d5jvm25b.dot (18 nodes)
Image renderer (dot) not found, not doing anything else


In [118]:
import tracemalloc

# A global list simulating an unbounded cache in a web application
_GLOBAL_CACHE = []


def process_request(data_payload):
    # Simulate processing logic
    processed_data = data_payload.upper()

    # MISTAKE: We are caching data without a limit or eviction policy.
    # This will slowly consume memory over the lifetime of the application.
    _GLOBAL_CACHE.append(processed_data)
    return processed_data


def simulate_traffic():
    # 1. Start tracing memory allocations.
    # The '10' stores up to 10 frames of traceback for precision.
    tracemalloc.start(10)

    # 2. Take our baseline snapshot BEFORE the traffic starts
    snapshot1 = tracemalloc.take_snapshot()

    # Simulate 100,000 requests coming into our backend
    for i in range(100000):
        process_request(f"request_data_{i}")

    # 3. Take our second snapshot AFTER the traffic
    snapshot2 = tracemalloc.take_snapshot()

    # 4. Compare the snapshots to see what grew
    # 'lineno' groups the results by the line of code that allocated the memory
    top_stats = snapshot2.compare_to(snapshot1, "lineno")

    print("[Top 3 Memory Allocations]")
    for stat in top_stats[:3]:
        print(stat)

    # Always stop tracemalloc when done to avoid its own performance overhead
    tracemalloc.stop()


simulate_traffic()

[Top 3 Memory Allocations]
/var/folders/cb/by3krgzd6rzgcyscfhww7j180000gn/T/ipykernel_2275/2524199403.py:9: size=5751 KiB (+5751 KiB), count=100000 (+100000), average=59 B
/var/folders/cb/by3krgzd6rzgcyscfhww7j180000gn/T/ipykernel_2275/2524199403.py:13: size=782 KiB (+782 KiB), count=1 (+1), average=782 KiB
/Users/riyaz/.local/share/uv/python/cpython-3.14.6-macos-aarch64-none/lib/python3.14/tracemalloc.py:560: size=328 B (+328 B), count=1 (+1), average=328 B


In [119]:
def greet():
    print("Hello Riyaz")

In [120]:
def audit_log(func):
    def wrapper():
        print(f"{func} is getting called...")
        func()
        print(f"{func} is completed..")

    return wrapper

In [121]:
log_greet = audit_log(greet)
log_greet()

<function greet at 0x110070930> is getting called...
Hello Riyaz
<function greet at 0x110070930> is completed..


In [122]:
import functools


def audit_log(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"{func} is getting called...")
        result = func(*args, **kwargs)
        print(f"{func} is completed..")
        return result

    return wrapper

In [123]:
@audit_log
def greet(name):
    print(f"Hello {name}")

In [124]:
greet("Riyaz")

<function greet at 0x1100716f0> is getting called...
Hello Riyaz
<function greet at 0x1100716f0> is completed..


In [125]:
import time


def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        res = func(*args, **kwargs)
        totl_time = time.perf_counter() - start
        print(f"Time took to complete: {totl_time:.2f} sec")
        return res

    return wrapper

In [126]:
@timer
def sum_of(num):
    total = 0
    for i in range(num):
        total += i
    return total

In [127]:
print(f"Sum of 10_00_00_000 is: {sum_of(10_00_00_000)}")

Time took to complete: 1.48 sec
Sum of 10_00_00_000 is: 4999999950000000


In [128]:
@timer
def fast_sum(num):
    return sum(list(range(num)))

In [129]:
print(f"Fast Sum, not that fast of 10_00_00_000 is: {fast_sum(10_00_00_000)}")

Time took to complete: 1.52 sec
Fast Sum, not that fast of 10_00_00_000 is: 4999999950000000


In [130]:
def cache(func):
    cached_data = {}

    @functools.wraps(func)
    def wrapper(num):
        start = time.perf_counter()
        res = None
        if num in cached_data:
            res = cached_data[num]
        if not res:
            res = func(num)
            cached_data[num] = res
        time_took = time.perf_counter() - start
        print(f"Time took to compute: {time_took:.6f}")
        return res

    return wrapper

In [131]:
@cache
def power_num(num):
    return num**num

In [132]:
print(f"Power of 100 is: {power_num(100)}")

Time took to compute: 0.000002
Power of 100 is: 100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000


In [133]:
def custom_for(numbers):
    itertor = iter(numbers)
    while True:
        try:
            val = next(itertor)
            print(val)
        except StopIteration:
            break

In [134]:
custom_for([1, 5, 3, 8])

1
5
3
8


In [138]:
a = [1, 2, 3]
# Here b creates a reference to a, but not copy it
b = a

c = a[:]

d = list(c)

print(f"Object refernce to a is: {sys.getrefcount(a) - 1}")
print(f"Object reference to c is: {sys.getrefcount(c) - 1}")
print(f"values in d: {d}")
print(f"Object refernce to d is: {sys.getrefcount(d) - 1}")

Object refernce to a is: 2
Object reference to c is: 1
values in d: [1, 2, 3]
Object refernce to d is: 1


In [140]:
class A:
    def process(self):

        print("A")


class B(A):
    def process(self):

        print("B")

        super().process()


class C(A):
    def process(self):

        print("C")

        super().process()


class D(B, C):
    def process(self):

        print("D")

        super().process()


application = D()

application.process()

D.mro()

D
B
C
A


[__main__.D, __main__.B, __main__.C, __main__.A, object]